# Prompt engineering

## Plan

### 1. Define correction scope.
- What errors should LLM generate - grammar, orthography, punctuation, style or all. 

### 2. Build a baseline zero-shot prompt.
- Example: "Create new sentence with simillar grammatical errors as in the following sentence. Return generated sentence only and nothing else."

### 3. Design few-shot examples stratified by error type.
- Cover major ERRANT categories: subject-verb agreement, article errors, prepositions, verb tense, spelling.
- 3-5 examples.
- Try balanced vs unbalanced error-type coverage as an ablation variable.

### 4. Add chain-of-thought variants.
- Ask the model to first identify the error type and then generate errored sentence.
- Compare CoT against direct-answer on precision and hallucination rate (over-correction).
- Expectations: improved precision, but slowed recall.

### 5. Systematic ablation.
- Change one variable at a time - instruction wording, number of examples, output format(sentence vs sentence + diffs), temperature.
- Log everything to create a prompt leaderboard based on ERRANT F0.5

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

/home/denispng/Code/tp_venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
load_dotenv()

True

In [ ]:
from langchain_openai import ChatOpenAI

# Free models on OpenRouter — swap if one is unavailable:
# "google/gemma-3-4b-it:free"
# "mistralai/mistral-7b-instruct:free"
# "microsoft/phi-3-mini-128k-instruct:free"
# Full list: openrouter.ai/models?q=:free

llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

In [3]:
# Load error types
import json
with open("gec_error_types.json") as f:
    error_types = json.load(f)["error_types"]

In [ ]:
# Make baseline with one-shot prompting.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

one_shot_prompt = ChatPromptTemplate.from_messages([
    ("human",
    "You are a grammatical error generator. "
    "For each sentence below, write a completely NEW and different sentence that contains the same type of grammatical error. "
    "Do NOT rewrite or correct the given sentence — create an entirely new one on a different topic. "
    "Return only the generated sentences, one per line, preserving the original numbering.\n\n"
    "Example:\n"
    "Input:  1. The group of students are going to the library.\n"
    "Output: 1. The team of engineers was arguing about their blueprints.\n\n"
    "{batch_input}"),

])

one_shot_chain = one_shot_prompt | llm | StrOutputParser()

In [ ]:
# Sample random examples from datasets
import glob

def sample_from_tsv(pattern, n_rows, seed=42, chunksize=5000):
    files = sorted(glob.glob(pattern))
    if not files:
        print("Wrong path or file name.")
        return []

    chunks = []
    for path in tqdm(files, desc="Sampling files"):
        reservoir = []
        for chunk in pd.read_csv(
            path, sep="\t", header=None,
            names=['original', 'ground_truth'],
            chunksize=chunksize
        ):
            reservoir.append(chunk.sample(n=min(n_rows, len(chunk)), random_state=seed))
        file_sample = pd.concat(reservoir).sample(n=n_rows, random_state=seed).reset_index(drop=True)
        chunks.append(file_sample)
        del reservoir, file_sample

    return pd.concat(chunks, ignore_index=True)

In [ ]:
sampled = sample_from_tsv("data/C4_200M.tsv-*", 100)

In [ ]:
test_sentences = sampled["original"]
ground_truth = sampled["ground_truth"]

In [ ]:
# One-shot baseline generation
import time
import re
import json as _json
from openai import RateLimitError

BATCH_SIZE = 5
SLEEP_BETWEEN_BATCHES = 10  # seconds between successful batches

def make_batch_input(sentences):
    return "\n".join(f"{i+1}. {s}" for i, s in enumerate(sentences))

def parse_batch_output(text, expected):
    # Primary: numbered lines (1. / 1) format)
    lines = [
        re.sub(r"^\d+[\.\)]\s*", "", l).strip()
        for l in text.strip().splitlines()
        if re.match(r"^\d+[\.\)]", l.strip())
    ]
    if len(lines) == expected:
        return lines
    # Fallback: model returned correct number of plain (unnumbered) lines
    fallback = [l.strip() for l in text.strip().splitlines() if l.strip()]
    if len(fallback) == expected:
        return fallback
    raise ValueError(f"Expected {expected} lines, got {len(lines)} numbered / {len(fallback)} plain:\n{text}")

def _is_provider_error(text):
    """Return True when the model body is actually a provider error dict string."""
    try:
        obj = _json.loads(text.replace("'", '"'))
        return "code" in obj and "message" in obj
    except Exception:
        return False

def _retry_after(e):
    """Extract Retry-After seconds from a RateLimitError, default 60s."""
    try:
        val = e.response.headers.get("Retry-After") or e.response.headers.get("x-ratelimit-reset-requests")
        if val:
            return int(float(val))
    except Exception:
        pass
    return 60

def invoke_batch_with_retry(chain, sentences, max_retries=7, base_delay=10):
    batch_input = make_batch_input(sentences)
    last_error = None
    for attempt in range(max_retries):
        try:
            result = chain.invoke({"batch_input": batch_input})
            if _is_provider_error(result):
                wait = 60 * (2 ** attempt)
                print(f"\n[Attempt {attempt+1}] Provider error, retrying in {wait}s: {result}")
                time.sleep(wait)
                continue
            return parse_batch_output(result, len(sentences))
        except ValueError as e:
            last_error = e
            wait = base_delay * (2 ** attempt)
            print(f"\n[Attempt {attempt+1}] Parse error: {e}")
            time.sleep(wait)
        except RateLimitError as e:
            if "per-day" in str(e) or "per_day" in str(e):
                raise RuntimeError(f"Daily request limit reached. Add credits or wait until tomorrow.\n{e}") from e
            wait = _retry_after(e)
            print(f"\nRate limit hit, retrying in {wait}s...")
            time.sleep(wait)
        except Exception as e:
            raise
    raise RuntimeError(f"Failed after {max_retries} retries. Last error: {last_error}")

sentences_list = test_sentences.tolist()
batches = [sentences_list[i:i+BATCH_SIZE] for i in range(0, len(sentences_list), BATCH_SIZE)]


In [ ]:

one_shot_generated = []
for batch in tqdm(batches, desc="One-shot baseline"):
    results = invoke_batch_with_retry(one_shot_chain, batch)
    one_shot_generated.extend(results)
    time.sleep(SLEEP_BETWEEN_BATCHES)

baseline_df = pd.DataFrame({
    "original": test_sentences.values,
    "generated": one_shot_generated,
    "ground_truth": ground_truth.values,
})

In [ ]:
baseline_df.to_csv("one_shot_baseline.csv", index=False)

In [ ]:
# Few-shot generation — stratified by error type (Point 3)
# One example per error type from gec_few_shot_examples.json → 12 examples total

with open("gec_few_shot_examples.json") as f:
    few_shot_data = json.load(f)

demonstrations = few_shot_data["demonstrations"]
error_types_order = json.load(open("gec_error_types.json"))["error_types"]

# few_shot_pairs: worked examples embedded in the prompt to teach the model the task.
# Each tuple is (incorrect_input, new_incorrect_output) — one per error type.
# Loaded from the "demonstrations" section of gec_few_shot_examples.json.
few_shot_pairs = [
    (demonstrations[et["name"]]["input"], demonstrations[et["name"]]["output"])
    for et in error_types_order
]

def make_few_shot_block(pairs):
    lines = []
    for i, (inp, out) in enumerate(pairs, 1):
        lines.append(f"Example {i}:")
        lines.append(f"Input:  {inp}")
        lines.append(f"Output: {out}")
        lines.append("")
    return "\n".join(lines)

few_shot_block = make_few_shot_block(few_shot_pairs)

few_shot_prompt = ChatPromptTemplate.from_messages([
    ("human",
     "You are a grammatical error generator. "
     "For each sentence below, write a completely NEW and different sentence that contains the same type of grammatical error. "
     "Do NOT rewrite or correct the given sentence — create an entirely new one on a different topic.\n"
     "Return only the generated sentences, one per line, preserving the original numbering.\n\n"
     f"{few_shot_block}"
     "Now generate for the following:\n"
     "{batch_input}"),
])

few_shot_llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

few_shot_chain = few_shot_prompt | few_shot_llm | StrOutputParser()

# Smaller batch size: the prompt is already long (12 few-shot examples),
# so fewer sentences per call reduces the risk of timeouts (524) and token-limit errors.
FS_BATCH_SIZE = 3
# Free-tier models have low RPM limits — sleep longer between batches to avoid rate limits.
FS_SLEEP_BETWEEN_BATCHES = 30

In [ ]:

# Sample from C4_200M — same approach as the one-shot baseline
fs_sampled = sample_from_tsv("data/C4_200M.tsv-*", 100)
fs_sentences = fs_sampled["original"]
fs_ground_truth = fs_sampled["ground_truth"]

fs_sentences_list = fs_sentences.tolist()
fs_batches = [fs_sentences_list[i:i+FS_BATCH_SIZE] for i in range(0, len(fs_sentences_list), FS_BATCH_SIZE)]


In [ ]:

few_shot_generated = []
for batch in tqdm(fs_batches, desc="Few-shot generation"):
    results = invoke_batch_with_retry(few_shot_chain, batch)
    few_shot_generated.extend(results)
    time.sleep(FS_SLEEP_BETWEEN_BATCHES)

few_shot_df = pd.DataFrame({
    "original": fs_sentences.values,
    "generated": few_shot_generated,
    "ground_truth": fs_ground_truth.values,
})

In [ ]:

few_shot_df.to_csv("few_shot_examples.csv", index=False)
few_shot_df.head()

In [9]:
# GEC correction of generated sentences using T5
# T5ForConditionalGeneration is seq2seq — must use AutoModelForSeq2SeqLM directly,
# not the text-generation pipeline (which only supports causal LMs in transformers 5.x).

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

one_shot_df = pd.read_csv("one_shot_baseline.csv")
few_shot_df  = pd.read_csv("few_shot_examples.csv")

model_name = "vennify/t5-base-grammar-correction"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()

def correct_sentences(sentences, batch_size=4, prefix="grammar: "):
    results = []
    for i in tqdm(range(0, len(sentences), batch_size), desc="T5 GEC"):
        batch = [prefix + s for s in sentences[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=128)
        results.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
    return results

for name, df in [("one_shot", one_shot_df), ("few_shot", few_shot_df)]:
    print(f"\nRunning T5 GEC on {name} ({len(df)} sentences)...")
    df["t5_corrected"] = correct_sentences(df["generated"].tolist())

one_shot_df.to_csv("one_shot_baseline_corrected.csv", index=False)
few_shot_df.to_csv("few_shot_examples_corrected.csv", index=False)

print("\nDone. Sample from few_shot:")
few_shot_df[["generated", "t5_corrected", "ground_truth"]].head(3)

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 30418.09it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Running T5 GEC on one_shot (1000 sentences)...


T5 GEC: 100%|██████████| 250/250 [06:03<00:00,  1.46s/it]



Running T5 GEC on few_shot (1000 sentences)...


T5 GEC: 100%|██████████| 250/250 [05:21<00:00,  1.29s/it]


Done. Sample from few_shot:


,generated,t5_corrected,ground_truth
0,"or perhaps we could meet later, at the cafe etc.","Or perhaps we could meet later, at the cafe, etc.","or part of it, to a garage, etc."
1,He installed latest version of the software on...,He installed the latest version of the softwar...,"Quick question, is it possible to run as non-r..."
2,"She likes reading novels, watching movies, and...","She likes reading novels, watching movies, and...",The Netflix documentary “Fyre” documents the f...
